# Matrices and Transformations

Companion notebook for the [Matrices and Transformations](https://ml-viz-ruby.vercel.app/courses/linear-algebra/02-matrices-and-transformations) lesson.

We'll visualize matrices as geometric transformations and explore matrix multiplication.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Visualizing linear transformations

We transform a unit square to see what each matrix does geometrically.

In [ ]:
def draw_transform(ax, A, title):
    """Draw the unit square and its image under matrix A."""
    # Unit square corners
    square = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]]).T   # 2×5
    transformed = A @ square

    ax.plot(*square,  color='#6366f1', lw=2, label='Original', alpha=0.7)
    ax.plot(*transformed, color='#f97316', lw=2, label='Transformed')

    # Basis vectors
    for vec, color in [([1,0], '#2dd4bf'), ([0,1], '#f59e0b')]:
        ax.annotate('', xy=vec, xytext=[0,0],
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.7))
        transformed_vec = A @ np.array(vec, dtype=float)
        ax.annotate('', xy=transformed_vec, xytext=[0,0],
                    arrowprops=dict(arrowstyle='->', color=color, lw=2))

    lim = max(3, abs(transformed).max() + 0.5)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.axhline(0, color='#30344a'); ax.axvline(0, color='#30344a')
    ax.set_title(title, pad=10)
    ax.legend(fontsize=9)

transforms = [
    (np.array([[2,0],[0,1]]), 'Scale x by 2'),
    (np.array([[0,-1],[1,0]]), 'Rotate 90°'),
    (np.array([[1,0],[0,0]]), 'Project onto x-axis'),
    (np.array([[1,1],[0,1]]), 'Shear'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (A, title) in zip(axes, transforms):
    draw_transform(ax, A, title)
plt.suptitle('Linear Transformations of the Unit Square', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Matrix multiplication as composition of transformations

In [ ]:
# Scale first, then rotate
scale  = np.array([[2, 0], [0, 0.5]])
rotate = np.array([[0, -1], [1, 0]])    # 90° counterclockwise

# Apply scale THEN rotate: C = R @ S
C = rotate @ scale

x = np.array([1.0, 1.0])
step1 = scale @ x           # after scaling
step2 = rotate @ step1      # after rotation
direct = C @ x              # direct composition

print('Original x:', x)
print('After scale:', step1)
print('After rotate (scale first):', step2)
print('Direct C @ x:', direct)
print('Are they equal?', np.allclose(step2, direct))

# Matrix multiplication is NOT commutative
print('\nR @ S =', C)
print('S @ R =', scale @ rotate)
print('Equal?', np.allclose(C, scale @ rotate))

## Rank and invertibility

In [ ]:
matrices = {
    'Full rank (invertible)': np.array([[3., 1.], [2., 4.]]),
    'Rank deficient (singular)': np.array([[1., 2.], [2., 4.]]),  # row 2 = 2 × row 1
    'Identity': np.eye(2),
}

for name, A in matrices.items():
    rank = np.linalg.matrix_rank(A)
    det = np.linalg.det(A)
    print(f'\n{name}')
    print(f'  Rank: {rank}')
    print(f'  Determinant: {det:.4f}')
    if abs(det) > 1e-10:
        print(f'  Inverse (first row): {np.linalg.inv(A)[0].round(4)}')
    else:
        print('  Not invertible (det ≈ 0)')

## The determinant as area scaling

The determinant of $\begin{bmatrix}a&b\\c&d\end{bmatrix}$ is $ad - bc$. Geometrically it is the **signed factor by which the map scales area**: the unit square becomes a parallelogram of area $|\det|$, with a negative sign when orientation flips. When $\det = 0$ the square collapses to a line — the transformation loses a dimension and is therefore not invertible. Below we verify $|\det|$ equals the image area computed independently with the shoelace formula.

In [ ]:
# The determinant = signed area-scaling factor.
# For a 2x2 matrix [[a,b],[c,d]] the formula is det = ad - bc.
def det2x2(M):
    (a, b), (c, d) = M
    return a * d - b * c

# Shoelace formula: area of the polygon the unit square maps to.
def polygon_area(pts):  # pts: 2 x N (last point need not repeat)
    x, y = pts
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

unit_square = np.array([[0, 1, 1, 0], [0, 0, 1, 1]], dtype=float)  # 4 corners

for name, A in [('scale 2x3', np.array([[2., 0.], [0., 3.]])),
                ('rotate 90', np.array([[0., -1.], [1., 0.]])),
                ('shear',     np.array([[1., 1.], [0., 1.]])),
                ('collapse',  np.array([[1., 2.], [2., 4.]]))]:
    image = A @ unit_square
    manual = det2x2(A)
    print(f'{name:10s}: det(ad-bc)={manual:+.1f}  np.det={np.linalg.det(A):+.1f}  '
          f'image area={polygon_area(image):.1f}  (=|det|={abs(manual):.1f})')
# 'collapse' has det 0 -> the unit square flattens to a line (area 0) -> not invertible.

## Visualizing a matrix transformation — before and after

The side-by-side plot below shows the unit square **before** (left) and **after** (right) a rotation+scaling matrix is applied, making the geometric effect of the transformation concrete.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Rotation by 45° composed with scaling (x by 1.5, y by 0.75)
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.array([[1.5, 0.0],
              [0.0, 0.75]])
A = R @ S   # scale first, then rotate

# Unit square corners (close the loop)
square = np.array([[0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0]], dtype=float)
transformed = A @ square

# Basis vectors before / after
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])
Ae1, Ae2 = A @ e1, A @ e2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

def draw_square_and_basis(ax, sq, b1, b2, title, sq_color='#6366f1',
                          c1='#2dd4bf', c2='#f97316'):
    ax.fill(sq[0], sq[1], color=sq_color, alpha=0.25)
    ax.plot(sq[0], sq[1], color=sq_color, lw=2)
    for vec, col, lbl in [(b1, c1, r'$\mathbf{e}_1$'), (b2, c2, r'$\mathbf{e}_2$')]:
        ax.annotate('', xy=vec, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=col, lw=2.2))
        ax.text(vec[0] + 0.05, vec[1] + 0.05, lbl, color=col, fontsize=12)
    ax.set_xlim(-2.0, 2.0); ax.set_ylim(-2.0, 2.0)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.25)
    ax.axhline(0, color='#30344a', lw=0.8); ax.axvline(0, color='#30344a', lw=0.8)
    ax.set_title(title, pad=12)

# Before
draw_square_and_basis(axes[0], square, e1, e2,
                      'Before: unit square + standard basis')

# After
draw_square_and_basis(axes[1], transformed, Ae1, Ae2,
                      r'After: $A = R_{45°} \cdot S_{1.5 \times 0.75}$',
                      sq_color='#f97316')

# Annotate determinant
det_val = np.linalg.det(A)
axes[1].text(0.02, 0.97,
             f'det(A) = {det_val:.3f}  →  area scales by |det| = {abs(det_val):.3f}',
             transform=axes[1].transAxes, va='top', color='#f59e0b', fontsize=10)

plt.suptitle('Matrix Transformation: Before vs. After', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Rotation matrices

A counterclockwise rotation by $\theta$ is the matrix

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Build it, and the checks confirm two facts from the lesson: composing rotations **adds their angles** ($R(45°)R(45°) = R(90°)$), and rotations **preserve area** ($\det R = 1$).

In [ ]:
def rotation(theta_degrees):
    """2x2 matrix that rotates vectors counterclockwise by theta_degrees."""
    t = np.radians(theta_degrees)

    # TODO(you): build [[cos, -sin], [sin, cos]] (hint: np.cos(t), np.sin(t))
    return ...

In [ ]:
# Checks — run me
assert np.allclose(rotation(90) @ [1, 0], [0, 1]), "90° sends the x-axis to the y-axis"
assert np.allclose(rotation(45) @ rotation(45), rotation(90)), "composing rotations adds angles"
assert abs(np.linalg.det(rotation(30)) - 1) < 1e-12, "rotations preserve area (det = 1)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def rotation(theta_degrees):
    t = np.radians(theta_degrees)
    return np.array([[np.cos(t), -np.sin(t)],
                     [np.sin(t),  np.cos(t)]])
```

</details>

### Exercise 2 — The determinant as area scaling

For a 2×2 matrix $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$, the determinant is $ad - bc$, and $|\det A|$ is the factor by which $A$ scales **areas**. Compute it by hand — no `np.linalg.det` allowed.

In [ ]:
def area_scale(A):
    """Factor by which the transformation A scales areas (always >= 0)."""
    A = np.asarray(A, dtype=float)

    # TODO(you): the 2x2 determinant by hand: ad - bc
    det = ...

    # TODO(you): area scaling is its absolute value
    return ...

In [ ]:
# Checks — run me
assert area_scale([[2, 0], [0, 3]]) == 6, "scaling x by 2 and y by 3 scales area by 6"
assert area_scale([[1, 1], [0, 1]]) == 1, "shear slants the square but preserves its area"
assert area_scale([[-1, 0], [0, 1]]) == 1, "reflection flips orientation but keeps area"
assert area_scale([[1, 2], [2, 4]]) == 0, "a singular matrix collapses the plane onto a line"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def area_scale(A):
    A = np.asarray(A, dtype=float)
    det = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    return abs(det)
```

</details>